In [135]:
import os, math
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [136]:
PROJECT = Path.cwd().parents[0]
PROCESSED = PROJECT / 'data' / 'processed'
test_csv = pd.read_csv(PROCESSED / 'test.csv')
train_csv = pd.read_csv(PROCESSED / 'train.csv')
valid_csv = pd.read_csv(PROCESSED / 'valid.csv')
OUTPUT_CSV = PROJECT / "results" / "data"
OUTPUT_CSV.mkdir(parents=True, exist_ok=True)


In [137]:
target_col = "SOURCE"
print("🎯 Using target:", target_col)
def encode_target(series: pd.Series):
    series = series.astype(str)
    classes = sorted(series.unique())
    mapping = {cls: i for i, cls in enumerate(classes)}
    print("🔑 Target mapping:", mapping)
    return series.map(mapping).astype(int),mapping
y_train, target_mapping = encode_target(train_csv[target_col])
y_valid = valid_csv[target_col].astype(str).map(target_mapping).astype(int)
y_test = test_csv[target_col].astype(str).map(target_mapping).astype(int)

#X train bỏ cột target
X_train = train_csv.drop(columns=[target_col])
X_valid = valid_csv.drop(columns=[target_col])
X_test = test_csv.drop(columns=[target_col])
print("X shapes:", X_train.shape, X_valid.shape, X_test.shape)

# Encode nếu cột còn dạng object
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
for c in cat_cols:
    X_train[c] = X_train[c].astype(str)
    X_valid[c] = X_valid[c].astype(str)
    X_test[c] = X_test[c].astype(str)

X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_valid_enc = pd.get_dummies(X_valid, columns=cat_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

print("Encoded X shapes:", X_train_enc.shape, X_valid_enc.shape, X_test_enc.shape)


🎯 Using target: SOURCE
🔑 Target mapping: {'in': 0, 'out': 1}
X shapes: (3088, 10) (662, 10) (662, 10)
Encoded X shapes: (3088, 10) (662, 10) (662, 10)


## MAIN MODEL: XG BOOST

In [138]:
class XGBoostModel():
    ''' XGBoost model from Scratch'''
    def __init__(self, params, random_seed = None):
        self.params = defaultdict(lambda: None, params)
        # The rationales of sample taking for training
        self.subsample = self.params['subsample'] if self.params['subsample'] else 1.0
        self.learning_rate = self.params['learning_rate'] if self.params['learning_rate'] else 0.3
        self.base_prediction = self.params['base_score'] if self.params['base_score'] else 0.5
        self.max_depth = self.params['max_depth'] if self.params['max_depth'] else 5
        self.rng = np.random.default_rng(seed=random_seed)
    ''' X: ma trận đầu vào
        y: label thật
        objective: 3 hàm (gradient, hessian, loss)
        num_boost_ground: số lượng cây cần xây
        verbose: True -> in loss sau mỗi vòng boosting
    '''
    def fit(self, X, y, objective, num_boost_round, verbose=False, x_valid = None, y_valid = None, early_stopping_rounds=None):
        current_predictions = self.base_prediction * np.ones(shape = y.shape)
        best_valid_loss = float("inf")
        best_round = -1
        best_boosters = None
        # Lưu tất car các cây trong quá trình xây dựng
        self.boosters = []
        for i in range(num_boost_round):
            gradients = objective.gradient(y, current_predictions)
            hessian = objective.hessian(y, current_predictions)
            sample_idxs = None if self.subsample == 1.0 else self.rng.choice(len(y), size=math.floor(self.subsample * len(y)), replace=False)
            booster = TreeBooster(X, gradients, hessian, self.params, self.max_depth, sample_idxs)
            current_predictions += self.learning_rate * booster.predict(X)
            self.boosters.append(booster)
            train_loss = objective.loss(y, current_predictions)
            msg = f"[{i}] train loss = {train_loss:.6f}"
            if x_valid is not None and y_valid is not None:
                y_raw_valid = self.base_prediction + self.learning_rate * np.sum([b.predict(x_valid) for b in self.boosters],axis=0)
                valid_loss = objective.loss(y_valid, y_raw_valid)
                msg += f" | valid loss = {valid_loss:.6f}"
                if valid_loss < best_valid_loss:
                    best_valid_loss = valid_loss
                    best_round = i
                    best_boosters = list(self.boosters)
                if early_stopping_rounds is not None and best_round >= 0:
                    if i - best_round >= early_stopping_rounds:
                        if verbose:
                            print(msg)
                            print(
                                f"⏹ Early stopping tại vòng {i}, "
                                f"best round = {best_round} (valid loss = {best_valid_loss:.6f})"
                            )
                        break
            if verbose:
                print(msg)
        if best_boosters is not None:
            self.boosters = best_boosters
            self.best_iteration = best_round
        else:
            self.best_iteration = len(self.boosters) - 1
    def predict(self,X):
        return (self.base_prediction + self.learning_rate * np.sum([booster.predict(X) for booster in self.boosters],axis=0))

class TreeBooster():
    def __init__(self, X, g, h, params, max_depth, idxs=None):
        self.params = params
        self.max_depth = max_depth
        assert max_depth >= 0, 'max_depth must be >= 0'
        # Nếu node con có tổng Hessian quá nhỏ → coi như ít dữ liệu → không split.
        self.min_child_weight = params['min_child_weight'] if params['min_child_weight'] else 1.0
        # tránh w quá lớn
        self.reg_lambda = params['reg_lambda'] if params['reg_lambda'] else 1.0
        # gain < gamma → bỏ.
        self.gamma = params['gamma'] if params['gamma'] else 1.0
        # Tỉ lệ feature được lấy mẫu ngẫu nhiên tại mỗi node.
        self.colsample_bynode = params['colsample_bynode'] if params['colsample_bynode'] else 1.0
        # chuyển ve numpy array
        if isinstance(g, pd.Series): g = g.values
        if isinstance(h, pd.Series): h = h.values
        if idxs is None: idxs = np.arange(len(g))
        self.X, self.g, self.h, self.idxs = X, g, h, idxs
        # c là số feature
        self.n, self.c = len(idxs), X.shape[1]
        self.value = -g[idxs].sum() /  (h[idxs].sum() + self.reg_lambda)
        self.best_score_so_far = 0
        if self.max_depth > 0:
            self._maybe_insert_child_nodes()
    def _maybe_insert_child_nodes(self):
        for i in range(self.c): self._find_better_split(i)
        if self.is_leaf: return
        x = self.X.values[self.idxs,self.split_feature_idx]
        left_idx = np.nonzero(x <= self.threshold)[0]
        right_idx = np.nonzero(x > self.threshold)[0]
        self.left = TreeBooster(self.X, self.g, self.h, self.params, self.max_depth - 1, self.idxs[left_idx])
        self.right = TreeBooster(self.X, self.g, self.h, self.params, self.max_depth - 1, self.idxs[right_idx])
    @property
    def is_leaf(self): return self.best_score_so_far == 0

    def _find_better_split(self, feature_index):
        x = self.X.values[self.idxs,feature_index]
        g,h = self.g[self.idxs],self.h[self.idxs]
        sort_idx = np.argsort(x)
        sort_g, sort_h, sort_x = g[sort_idx], h[sort_idx], x[sort_idx]
        sum_g, sum_h = g.sum(), h.sum()
        sum_g_right, sum_h_right = sum_g, sum_h
        sum_g_left, sum_h_left = 0.,0.
        for i in range(0, self.n-1):
            g_i, h_i, x_i, x_i_next = sort_g[i], sort_h[i], sort_x[i], sort_x[i+1]
            sum_g_left += g_i; sum_g_right -= g_i
            sum_h_left += h_i; sum_h_right -= h_i
            # Nếu tổng hessian quá nhỏ hoặc 2 giá trị giống nhau
            if sum_h_left < self.min_child_weight or x_i == x_i_next: continue
            if sum_h_right < self.min_child_weight: break
            gain = 0.5 * ((sum_g_left**2 / (sum_h_left + self.reg_lambda))
                            + (sum_g_right**2 / (sum_h_right + self.reg_lambda))
                            - (sum_g**2 / (sum_h + self.reg_lambda))
                            ) - self.gamma/2
            if gain > self.best_score_so_far:
                self.split_feature_idx = feature_index
                self.best_score_so_far = gain
                self.threshold = (x_i + x_i_next) / 2

    def predict(self, X):
        return np.array([self._predict_row(row) for i, row in X.iterrows()])

    def _predict_row(self, row):
        if self.is_leaf: return self.value
        child = (self.left if row.iloc[self.split_feature_idx] <= self.threshold else self.right)
        return child._predict_row(row)




In [139]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

class LogisticObjective:
    @staticmethod
    def loss(y,y_raw):
        p = sigmoid(y_raw)
        eps = 1e-12
        return -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))

    #Đạo hàm của loss theo y_raw
    @staticmethod
    def gradient(y, y_raw):
        p = sigmoid(y_raw)
        return (p - y)

    @staticmethod
    def hessian(y, y_raw):
        p = sigmoid(y_raw)
        return p * (1.0 - p)




In [ ]:
# ==== Khởi tạo tham số & model XGBoost-from-scratch ====
pos_rate = float(y_train.mean())
pos_rate = max(min(pos_rate, 1.0 - 1e-6), 1e-6) # Tránh bằng 0 hoặc 1 tuyệt đối
base_score = math.log(pos_rate / (1 - pos_rate))
# sigmoid(base_score) = pos_rate -> giải ngược thi sẽ ra được basescore như vậy

params = {
    "learning_rate": 0.15,
    "max_depth": 4,
    "subsample": 0.9,
    "min_child_weight": 4.0, #nên thử từ 1 tới 5
    "reg_lambda": 2.0,
    "gamma": 2.0, #nên thu từ 1-5
    "base_score": base_score,
    "colsample_bynode": 0.9,
}
objective = LogisticObjective()
model = XGBoostModel(params=params, random_seed=42)

num_boost_rounds = 200
print("🚀 Training XGBoost-from-scratch with target = SOURCE ...")
model.fit(
    X_train_enc,
    y_train,
    objective=objective,
    num_boost_round=num_boost_rounds,
    verbose=True,
    x_valid=X_valid_enc,
    y_valid=y_valid,
    early_stopping_rounds=40,
)

def evaluate_split(X, y_true, split_name="Valid"):
    y_raw = model.predict(X)
    y_prob = sigmoid(y_raw)
    y_pred = (y_prob >= 0.5).astype(int)

    acc = (y_pred == y_true).mean()
    print(f"\n {split_name} accuracy: {acc:.4f}")
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    print(f"{split_name} confusion matrix (tn, fp, fn, tp):")
    print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    return y_pred, y_prob
y_valid_pred, y_valid_prob = evaluate_split(X_valid_enc, y_valid, split_name="Valid")
y_test_pred,  y_test_prob  = evaluate_split(X_test_enc,  y_test,  split_name="Test")

inv_mapping = {v: k for k, v in target_mapping.items()}

valid_pred_labels = [inv_mapping[int(i)] for i in y_valid_pred]
test_pred_labels  = [inv_mapping[int(i)] for i in y_test_pred]

test_file  = OUTPUT_CSV / "xgb_pred_test.csv"
test_out = test_csv.copy()
test_out[target_col] = [inv_mapping[int(i)] for i in y_test_pred]
test_out.to_csv(test_file, index=False, encoding="utf-8-sig")

print("\n✅ Đã lưu file dự đoán:")
print("  •", test_file)



🚀 Training XGBoost-from-scratch with target = SOURCE ...
[0] train loss = 0.640575 | valid loss = 0.653001
[1] train loss = 0.613948 | valid loss = 0.628721
[2] train loss = 0.593577 | valid loss = 0.608344
[3] train loss = 0.577711 | valid loss = 0.596094
[4] train loss = 0.564152 | valid loss = 0.583328
[5] train loss = 0.552888 | valid loss = 0.573369
[6] train loss = 0.544014 | valid loss = 0.565494
[7] train loss = 0.536332 | valid loss = 0.558586
[8] train loss = 0.530187 | valid loss = 0.555444
